**Before starting this practical session**:

1. **Change the runtime type on Google Colab**: `Runtime > Change runtime type > T4 GPU`
2. **Install the packages below**:

In [1]:
! pip install lightning torchmetrics torchinfo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 37.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 52.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 42.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [lightning]


3. Run this code to suppress some messages and warnings that may be displayed.

In [2]:
import logging
logging.getLogger("lightning").setLevel(logging.ERROR)
logging.getLogger("lightning.pytorch.utilities.rank_zero").setLevel(logging.WARNING)
logging.getLogger("lightning.pytorch.accelerators.cuda").setLevel(logging.WARNING)
logger = logging.getLogger("lightning")
logger.propagate = False

import warnings
warnings.filterwarnings("ignore", ".*does not have many workers.*")
warnings.filterwarnings("ignore", ".*Missing logger folder.*")
warnings.filterwarnings("ignore", "`isinstance\\(treespec, LeafSpec\\)` is deprecated.*")

# My first artificial neural network

During the second part of this first practical session, you will work on another dataset: [*forest cover types*](https://archive.ics.uci.edu/dataset/31/covertype).
The goal is to predict the type of a forest tree from certain characteristics.
This is a **classification** problem.

Using what you learned in the previous practical session, you will need to:

* **preprocess the data**,
* **specify how to access the data**,
* **build a neural network**,
* **train and evaluate this neural network**

We will use the `scikit-learn` package to download this dataset.

We will (down)load this dataset using the [`sklearn.datasets.fetch_covtype()`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_covtype.html) function.
In summary, this function returns two variables:

* `X` is a matrix (a two-dimensional NumPy array) of size $n \times p$ where $n$ is the number of observations and $p$ is the number of variables. This is the input data.
* `y` is a vector (a one-dimensional NumPy array) of size $n$. This is the output data (to be predicted).

In [3]:
from sklearn.datasets import fetch_covtype

X, y = fetch_covtype(data_home='data', return_X_y=True)

### Question 1

1. Determine the size of the dataset, i.e., the number of observations $n$ and the number of variables $p$. You can use the [`numpy.ndarray.shape`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.shape.html) attribute.

2. Determine the number of classes. Are the classes balanced? You can use the [`numpy.unique()`](https://numpy.org/doc/stable/reference/generated/numpy.unique.html) function.

In [10]:
import numpy as np

n, p = X.shape

print(f"num. observations: {n} \nnum. of variables {p}")

class_labels, class_counts = np.unique(y, return_counts = True)
print(class_labels, class_counts)

num. observations: 581012 
num. of variables 54
[1 2 3 4 5 6 7] [211840 283301  35754   2747   9493  17367  20510]


There are seven classes.
No, the classes are not balanced.

### Question 2

Split the dataset into three sets:
* a training set with 100,000 observations,
* a validation set with 100,000 observations,
* a test set with the remaining observations.

You can use the [`sklearn.model_selection.train_test_split()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function.
Make sure the class distribution is identical across the three subsets by using the `stratify` parameter.

In [20]:
from sklearn.model_selection import train_test_split

X_notest, X_test, y_notest, y_test = train_test_split(X, y, train_size=200000, stratify=y, random_state=1234) #We stratify by class to make sure that all classes are properly represented in train and test
X_train, X_val, y_train, y_val = train_test_split(X_notest, y_notest, train_size=100_000, stratify=y_notest, random_state=1234)

### Question 3

Convert the NumPy arrays to PyTorch tensors. Don't forget to change the data type:
* the input data (`X`) must go from `numpy.float64` to `torch.float32`,
* the output data (`y`) must go from `numpy.int32` to `torch.int64`.

You can use the [`torch.from_numpy()`](https://pytorch.org/docs/stable/generated/torch.from_numpy.html) function and the [`torch.Tensor.to()`](https://pytorch.org/docs/stable/generated/torch.Tensor.to.html) method.

> **Note**: For classification tasks, it is necessary to provide a suitable representation of the classes for the loss function. The simplest representation for a multi-class classification task is to use the first $K$ natural integers (starting from zero), i.e., the integers $0, \ldots, K-1$, where $K$ is the number of classes. This way, the correspondence between the last layer of the neural network (returning probabilities or logits) and the classes is based on indices: `probability[k]` corresponds to the probability of belonging to class $k$ for each $k \in \{ 0, \ldots, K-1 \}$.

In [21]:
import torch

X_train_tensor = torch.from_numpy(X_train).to(dtype=torch.float32)
X_val_tensor = torch.from_numpy(X_val).to(dtype=torch.float32)
X_test_tensor = torch.from_numpy(X_test).to(dtype=torch.float32)

y_train_tensor = torch.from_numpy(y_train-1).to(dtype=torch.int64)
y_val_tensor = torch.from_numpy(y_val-1).to(dtype=torch.int64)
y_test_tensor = torch.from_numpy(y_test-1).to(dtype=torch.int64)

### Question 4

Create instances of the [`torch.utils.data.DataLoader()`](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) class for each of the sets (training, validation, and test). For the datasets, you can use the [`torch.utils.data.TensorDataset()`](https://pytorch.org/docs/stable/data.html#torch.utils.data.TensorDataset) class.

In [22]:
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

#Defining dataset class
dataloader_train = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=100,
    shuffle=True
    )

dataloader_val = DataLoader(
    TensorDataset(X_val_tensor, y_val_tensor),
    batch_size=100,
    shuffle=False
    )

dataloader_test = DataLoader(
    TensorDataset(X_test_tensor, y_test_tensor),
    batch_size=100,
    shuffle=False
    )

### Question 5

The definition of the model's main characteristics (architecture) and its training (optimization algorithm, loss function, evaluation metric) is done in the same class.
Complete the `__init__()`, `forward()`, and `configure_optimizers()` methods of the `NeuralNetwork` class defined below using the information provided in the text below.

#### Architecture

The architecture of your neural network is a **multilayer perceptron** with the following characteristics:
* *First hidden layer*: linear layer (128 output variables) + ReLU activation function
* *Second hidden layer*: linear layer (64 output variables) + ReLU activation function
* *Output layer*: linear layer (up to you to determine the output size)

As a reminder, layers are initialized in the constructor and the forward pass is defined in the `forward()` method.
You are encouraged to read the documentation of [`torch.nn.Linear()`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html), [`torch.nn.ReLU()`](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html), and [`torch.nn.Sequential()`](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html).

#### Training

The model will be trained using cross-entropy as the loss function and Adam with default values for its hyperparameters as the optimization algorithm.
You are encouraged to read the documentation of [`torch.nn.CrossEntropyLoss()`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) and [`torch.optim.Adam()`](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html).

#### Metric

The performance of a model will be evaluated using accuracy.
You can use [`torchmetrics.Accuracy()`](https://lightning.ai/docs/torchmetrics/stable/classification/accuracy.html).

In [25]:
import lightning as L
from torchmetrics import Accuracy


class NeuralNetwork(L.LightningModule):  # Always inherit from the lightning.LightningModule class

    def __init__(self):
        """Constructor.

        In the constructor, we always run the constructor of the parent class and we
        instantiate all the elements of our network: the layers, the activation functions,
        the loss function and the metrics.
        """
        super().__init__()  # Always run the constructor of the parent class

        ### BEGIN TODO ###
        # Instantiate the layers and activation functions
        self.sequential = torch.nn.Sequential(
            torch.nn.Linear(54, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 7)
        )

        # Instantiate the loss function
        self.loss = torch.nn.CrossEntropyLoss()

        # Instantiate the metrics
        self.accuracy_train = Accuracy(task="multiclass", num_classes=7)
        self.accuracy_val = Accuracy(task="multiclass", num_classes=7)
        self.accuracy_test = Accuracy(task="multiclass", num_classes=7)
        #### END TODO ####

    def forward(self, x):
        """Implements the forward pass."""
        ### BEGIN TODO ###
        y = self.sequential(x)
        #### END TODO ####
        return y

    def step(self, batch, dataset):
        """Perform a step.

        A step consists in mapping a batch of samples to the loss function for this batch of input samples.

        Parameters
        ----------
        batch : tuple
            A batch of samples.

        dataset : {"training", "validation"}
            Data set used.

        Returns
        -------
        loss : Tensor, shape = (1,)
            Loss function evaluated for this batch of samples.
        """
        x, y = batch  # x corresponds to the inputs, y corresponds to the classes (labels)
        logits = self(x)  # Forward pass outputting the logits
        loss = self.loss(logits, y)  # Evaluation of the loss function
        y_pred = logits.argmax(1)  # Predictions of the model

        if dataset == "training":
            metric = self.accuracy_train
            name = "train"
            bar_step = True
        elif dataset == "validation":
            metric = self.accuracy_val
            name = "val"
            bar_step = False
        else:
            metric = self.accuracy_test
            name = "test"
            bar_step = False

        acc = metric(y_pred, y)
        self.log(f"loss_{name}", loss, prog_bar=bar_step, on_step=bar_step, on_epoch=True)
        self.log(f"accuracy_{name}", acc, prog_bar=bar_step, on_step=bar_step, on_epoch=True)

        return loss

    def training_step(self, batch):
        """Perform a training step."""
        return self.step(batch, "training")

    def validation_step(self, batch):
        """Perform a validation step."""
        return self.step(batch, "validation")

    def test_step(self, batch):
        """Perform a test step."""
        return self.step(batch, "test")

    def on_train_start(self):
        """Code to run at the beginning of training."""
        string = f"Version {self.trainer.logger.version}"
        print(f"{string}\n{'=' * len(string)}\n")

    def on_train_epoch_end(self):
        """Code to run at the end of each training epoch."""
        metrics = self.trainer.callback_metrics
        string = (f"""
            Epoch {self.trainer.current_epoch + 1} / {self.trainer.max_epochs}
            -----------------------------------------
            |  Data set  | Loss function | Accuracy |
            | ---------- | ------------- | -------- |
            |  Training  |{metrics['loss_train'].item():^15.5f}|{metrics['accuracy_train'].item():^10.4%}|
            | Validation |{metrics['loss_val'].item():^15.5f}|{metrics['accuracy_val'].item():^10.4%}|
            -----------------------------------------
        """)
        string = "\n".join([line.strip() for line in string.strip().split("\n")])
        print(string, "\n")

    def configure_optimizers(self):
        """Configure the optimization algorithms."""
        ### BEGIN TODO ###
        optimizer = torch.optim.Adam(self.parameters())
        #### END TODO ####
        return optimizer

We will now train the model for 10 epochs.

In [26]:
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger


model = NeuralNetwork()

trainer = L.Trainer(
    max_epochs=10,
    enable_model_summary=False,  # disable model summary
    logger=CSVLogger('.'),  # save the results in a CSV file
    num_sanity_val_steps=0,  # do not perform any validation step before training
    callbacks=[TQDMProgressBar(refresh_rate=100)]  # update the progress bar every 100 batches
)

trainer.fit(
    model=model,
    train_dataloaders=dataloader_train,
    val_dataloaders=dataloader_val
)

Training: |          | 0/? [00:00<?, ?it/s]

Version 0



Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 1 / 10
-----------------------------------------
|  Data set  | Loss function | Accuracy |
| ---------- | ------------- | -------- |
|  Training  |    1.74871    | 54.5100% |
| Validation |    1.60671    | 48.4720% |
----------------------------------------- 



Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 2 / 10
-----------------------------------------
|  Data set  | Loss function | Accuracy |
| ---------- | ------------- | -------- |
|  Training  |    0.98299    | 62.0450% |
| Validation |    0.87478    | 63.4810% |
----------------------------------------- 



... (12 outputs left out)

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 9 / 10
-----------------------------------------
|  Data set  | Loss function | Accuracy |
| ---------- | ------------- | -------- |
|  Training  |    0.64830    | 72.0680% |
| Validation |    0.63810    | 72.8410% |
----------------------------------------- 



Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 10 / 10
-----------------------------------------
|  Data set  | Loss function | Accuracy |
| ---------- | ------------- | -------- |
|  Training  |    0.64416    | 72.0960% |
| Validation |    0.65137    | 71.8800% |
----------------------------------------- 



### Question 6

Is accuracy an appropriate metric here?
Which metric would be more relevant?
Are there also changes to make to potentially improve training?
Look at the documentation of [`torchmetrics.Accuracy()`](https://lightning.ai/docs/torchmetrics/stable/classification/accuracy.html) and [`torch.nn.CrossEntropyLoss()`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) and make the necessary changes in the `__init__()` method.
You can use the [`torch.bincount()`](https://pytorch.org/docs/stable/generated/torch.bincount.html) function to count the number of observations for the different classes. The constructor of the `NeuralNetworkUpdated()` class takes the labels of the training set (`y_train`) as argument.

> **Note**: Balanced accuracy requires knowing the class distribution in order to know the class weights. Since the class distribution is only known at the end (once the entire dataset has been traversed), it is not possible to compute balanced accuracy scores on all intermediate batches. The correct approach is to *update* the metric (with the `update()` method) at each step, then compute the balanced accuracy at the end of the epoch (with the `compute()` method), and finally reset the information saved under the hood in order to compute the balanced accuracy (with the `reset()` method).

In [ ]:
import lightning as L


class NeuralNetworkUpdated(L.LightningModule):  # Always inherit from the lightning.LightningModule class

    def __init__(self, y_train):
        """Constructor.

        In the constructor, we always run the constructor of the parent class and we
        instantiate all the elements of our network: the layers, the activation functions,
        the loss function and the metrics.

        Parameters
        ----------
        y_train : Tensor, shape = (n_samples,)
            Labels of the training set (used to compute the class weights).
        """
        super().__init__()  # Always run the constructor of the parent class

        self.sequential = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(54, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 7),
        )

        ### BEGIN TODO ###
        self.loss = torch.nn.CrossEntropyLoss(weight=torch.from_numpy(class_counts.sum()/class_counts).to(dtype=torch.float32))
        self.balanced_accuracy_train =
        self.balanced_accuracy_val =
        self.balanced_accuracy_test =
        #### END TODO ####

    def forward(self, x):
        """Implements the forward pass."""
        return self.sequential(x)

    def step(self, batch, dataset):
        """Perform a step.

        A step consists in mapping a batch of samples to the loss function for this batch of input samples.

        Parameters
        ----------
        batch : tuple
            A batch of samples.

        dataset : {"training", "validation"}
            Data set used.

        Returns
        -------
        loss : Tensor, shape = (1,)
            Loss function evaluated for this batch of samples.
        """
        x, y = batch  # x corresponds to the inputs, y corresponds to the classes (labels)
        logits = self(x)  # Forward pass outputting the logits
        loss = self.loss(logits, y)  # Evaluation of the loss function
        y_pred = logits.argmax(1)  # Predictions of the model

        if dataset == "training":
            metric = self.balanced_accuracy_train
            name = "train"
            bar_step = True
        elif dataset == "validation":
            metric = self.balanced_accuracy_val
            name = "val"
            bar_step = False
        else:
            metric = self.balanced_accuracy_test
            name = "test"
            bar_step = False

        # Only update the metric here: the balanced accuracy is computed (and logged)
        # at the end of each epoch using all the predictions of the epoch
        metric.update(y_pred, y)
        self.log(f"weighted_loss_{name}", loss, prog_bar=bar_step, on_step=bar_step, on_epoch=True)

        return loss

    def training_step(self, batch):
        """Perform a training step."""
        return self.step(batch, "training")

    def validation_step(self, batch):
        """Perform a validation step."""
        return self.step(batch, "validation")

    def test_step(self, batch):
        """Perform a test step."""
        return self.step(batch, "test")

    def on_train_start(self):
        """Code to run at the beginning of training."""
        string = f"Version {self.trainer.logger.version}"
        print(f"{string}\n{'=' * len(string)}\n")

    def on_train_epoch_end(self):
        """Code to run at the end of each training epoch."""
        self.log('balanced_accuracy_train', self.balanced_accuracy_train.compute())
        self.balanced_accuracy_train.reset()

        metrics = self.trainer.callback_metrics
        weighted_loss_train = metrics['weighted_loss_train'].item()
        weighted_loss_val = metrics['weighted_loss_val'].item()
        balanced_accuracy_train = metrics['balanced_accuracy_train'].item()
        balanced_accuracy_val = metrics['balanced_accuracy_val'].item()

        string = (f"""
            Epoch {self.trainer.current_epoch + 1} / {self.trainer.max_epochs}
            -----------------------------------------------------------
            |  Data set  | Weighted loss function | Balanced accuracy |
            | ---------- | ---------------------- | ----------------- |
            |  Training  |{weighted_loss_train:^24.4f}|{balanced_accuracy_train:^19.3%}|
            | Validation |{weighted_loss_val:^24.4f}|{balanced_accuracy_val:^19.3%}|
            -----------------------------------------------------------
        """)
        string = "\n".join([line.strip() for line in string.strip().split("\n")])
        print(string, "\n")

    def on_validation_epoch_end(self):
        """Code to run at the end of each validation epoch."""
        self.log('balanced_accuracy_val', self.balanced_accuracy_val.compute())
        self.balanced_accuracy_val.reset()

    def on_test_epoch_end(self):
        """Code to run at the end of each test epoch."""
        self.log('balanced_accuracy_test', self.balanced_accuracy_test.compute())
        self.balanced_accuracy_test.reset()

    def configure_optimizers(self):
        """Configure the optimization algorithms."""
        optimizer = torch.optim.Adam(self.parameters())
        return optimizer

We also train a new model with the new class for $10$ epochs.

In [ ]:
model_updated = NeuralNetworkUpdated(y_train)

trainer_updated = L.Trainer(
    max_epochs=10,
    enable_model_summary=False,  # disable model summary
    logger=CSVLogger('.'),  # save the results in a CSV file
    num_sanity_val_steps=0,  # do not perform any validation step before training
    callbacks=[TQDMProgressBar(refresh_rate=100)]  # update the progress bar every 100 batches
)

trainer_updated.fit(
    model=model_updated,
    train_dataloaders=dataloader_train,
    val_dataloaders=dataloader_val
)

### Question 7

Make the changes you want, for example to the architecture or the training procedure, and train your new models.
**Keep your previous models** and create new objects each time, so that you can compare these different models afterward.

In [ ]:
# TODO

### Question 8

When you have finished all your experiments, it is time to choose the best model on the validation set.
Evaluate its performance on the test set.
Out of curiosity, also evaluate the performance of the other models on the test set.
You are encouraged to read the [documentation](https://lightning.ai/docs/torchmetrics/stable/pages/overview.html) of `torchmetrics` to discover how to use the metrics implemented in this package.

> **Note**: The first class uses accuracy as the evaluation metric, while the second class uses balanced accuracy. It is obviously not relevant to compare accuracy scores with balanced accuracy scores. Similarly, the loss function is now weighted in the second class. It is therefore not possible to use the `validate()` and `test()` methods to compare models defined by different classes if the classes use different evaluation criteria.

In [ ]:
# TODO